# Extensión C: medir una asociación sin sobreinterpretarla (WEAT)

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Qué se puede afirmar, y qué no, cuando medimos si un conjunto de palabras queda más cerca de otro?

## Objetivo

Implementar la prueba WEAT (Caliskan y otros, 2017) con vectores de palabras en español, y comprobar cuánto cambia el resultado al cambiar los conjuntos de palabras.

## Al terminar este cuaderno vamos a poder:

1. Calcular el estadístico WEAT y su tamaño de efecto a partir de cuatro conjuntos de palabras definidos de antemano.
2. Reconocer que el resultado depende de las listas elegidas y que una sola prueba no establece discriminación.
3. Distinguir observación, hipótesis, auditoría y consecuencias de uso.

## Prerrequisitos reales

* Cuadernos 02 y 03 de esta unidad (vectores de palabras y vocabulario explícito).

## Recursos y condiciones de ejecución

Local. Carga los vectores Word2Vec (unos 2 GB de memoria).

## Qué es esencial y qué es opcional

* **Esencial:** Opcional.
* **Opcional:** Todo.

## Producto

Una tabla de tres variantes de la prueba con una conclusión de cinco líneas que separe lo observado de lo que se podría inferir.

## Alcance

Tres variantes de listas de ocho palabras por conjunto, armadas por nosotros. Es un ejercicio de lectura crítica, no una auditoría del modelo.

In [1]:
# Esta extensión vive en extensiones/. Subimos a la carpeta de la unidad para que las rutas relativas funcionen.
import os
from pathlib import Path

if Path.cwd().name == "extensiones":
    os.chdir(Path.cwd().parent)
print("Carpeta de trabajo:", Path.cwd().name)

Carpeta de trabajo: 009_representaciones_semanticas


## 1. Qué mide WEAT

Dos conjuntos de **palabras objetivo** (X e Y, por ejemplo nombres masculinos y femeninos) y dos de **atributos** (A y B, por ejemplo palabras de trabajo y de hogar). Para cada palabra objetivo, se calcula cuánto más cerca está de A que de B (la media de sus cosenos con A menos la media con B). El **tamaño de efecto** d compara los promedios de X e Y, dividido por la desviación estándar de todos los valores.

* d positivo: X queda más cerca de A que Y.
* d cercano a 0: no hay diferencia sistemática en esas listas.
* Con ocho palabras por conjunto, d varía mucho si cambia una sola palabra.

**Qué no hace.** No prueba que un sistema discrimine en una aplicación, ni mide el sesgo "del modelo" en general: mide una asociación entre **estas** listas en **este** espacio vectorial.

In [2]:
import sys
import importlib.metadata

import numpy as np
import pandas as pd
from IPython.display import display

import utils

SEMILLA = 42
print("Python:", sys.version.split()[0], "| gensim:", importlib.metadata.version("gensim"), "| semilla:", SEMILLA)

vectores = utils.cargar_vectores_word2vec()
print(f"Word2Vec: {len(vectores.key_to_index):,} palabras")

Python: 3.12.13 | gensim: 4.4.0 | semilla: 42


Word2Vec: 1,692,025 palabras


## 2. Los conjuntos, fijados antes de calcular

Armamos tres variantes. La primera es nuestra propuesta; las otras dos cambian **un solo conjunto** para ver cuánto pesa esa decisión. Los nombres propios se escriben con mayúscula inicial porque así están en el vocabulario de este modelo.

**Decisión previa.** No ajustamos las listas mirando los resultados.

In [3]:
nombres_masculinos = ["Juan", "Pedro", "Carlos", "Luis", "Miguel", "José", "Diego", "Martín"]
nombres_femeninos = ["María", "Ana", "Laura", "Lucía", "Carla", "Sofía", "Julia", "Paula"]

trabajo = ["trabajo", "oficina", "empresa", "salario", "carrera", "negocio", "profesional", "dirección"]
hogar = ["hogar", "familia", "hijos", "cocina", "casa", "niños", "matrimonio", "parientes"]

# Variante 2: mismos nombres, otras palabras de trabajo y de hogar.
trabajo_2 = ["ingeniería", "ciencia", "tecnología", "economía", "política", "industria", "mercado", "ejército"]
hogar_2 = ["arte", "poesía", "literatura", "música", "danza", "teatro", "pintura", "moda"]

# Variante 3: los atributos originales, con otros nombres.
nombres_masculinos_3 = ["Jorge", "Héctor", "Raúl", "Ricardo", "Alberto", "Hugo", "Oscar", "Rubén"]
nombres_femeninos_3 = ["Marta", "Susana", "Silvia", "Graciela", "Norma", "Rosa", "Elena", "Beatriz"]

variantes = [
    {"nombre": "1. propuesta inicial", "X": nombres_masculinos, "Y": nombres_femeninos, "A": trabajo, "B": hogar},
    {"nombre": "2. otros atributos", "X": nombres_masculinos, "Y": nombres_femeninos, "A": trabajo_2, "B": hogar_2},
    {"nombre": "3. otros nombres", "X": nombres_masculinos_3, "Y": nombres_femeninos_3, "A": trabajo, "B": hogar},
]

# Verificamos cobertura: cada palabra debe tener entrada explícita.
for variante in variantes:
    faltan = []
    for conjunto in ["X", "Y", "A", "B"]:
        for palabra in variante[conjunto]:
            if palabra not in vectores.key_to_index:
                faltan.append(palabra)
    print(variante["nombre"], "| palabras sin vector:", faltan)

1. propuesta inicial | palabras sin vector: []
2. otros atributos | palabras sin vector: []
3. otros nombres | palabras sin vector: []


## 3. El cálculo, paso a paso

In [4]:
def asociación_de_una_palabra(palabra, atributos_a, atributos_b):
    """Media del coseno con los atributos A menos la media con los atributos B."""
    cosenos_a = []
    for atributo in atributos_a:
        cosenos_a.append(utils.coseno(vectores[palabra], vectores[atributo]))
    cosenos_b = []
    for atributo in atributos_b:
        cosenos_b.append(utils.coseno(vectores[palabra], vectores[atributo]))
    return np.mean(cosenos_a) - np.mean(cosenos_b)


def tamaño_de_efecto(conjunto_x, conjunto_y, atributos_a, atributos_b):
    """Tamaño de efecto d de WEAT y los valores individuales de cada palabra."""
    valores_x = []
    for palabra in conjunto_x:
        valores_x.append(asociación_de_una_palabra(palabra, atributos_a, atributos_b))
    valores_y = []
    for palabra in conjunto_y:
        valores_y.append(asociación_de_una_palabra(palabra, atributos_a, atributos_b))
    todos = valores_x + valores_y
    d = (np.mean(valores_x) - np.mean(valores_y)) / np.std(todos, ddof=1)
    return d, valores_x, valores_y


def prueba_de_permutaciones(conjunto_x, conjunto_y, atributos_a, atributos_b, repeticiones=5000):
    """Proporción de reasignaciones al azar de las palabras objetivo con una diferencia al menos tan grande."""
    palabras_objetivo = conjunto_x + conjunto_y
    valores = []
    for palabra in palabras_objetivo:
        valores.append(asociación_de_una_palabra(palabra, atributos_a, atributos_b))
    valores = np.array(valores)
    diferencia_observada = valores[:len(conjunto_x)].mean() - valores[len(conjunto_x):].mean()

    generador = np.random.default_rng(SEMILLA)
    mayores_o_iguales = 0
    for repetición in range(repeticiones):
        mezcla = generador.permutation(valores)
        diferencia_al_azar = mezcla[:len(conjunto_x)].mean() - mezcla[len(conjunto_x):].mean()
        if abs(diferencia_al_azar) >= abs(diferencia_observada):
            mayores_o_iguales += 1
    return mayores_o_iguales / repeticiones


filas = []
for variante in variantes:
    d, valores_x, valores_y = tamaño_de_efecto(variante["X"], variante["Y"], variante["A"], variante["B"])
    p = prueba_de_permutaciones(variante["X"], variante["Y"], variante["A"], variante["B"])
    filas.append({"variante": variante["nombre"], "d (tamaño de efecto)": round(d, 2), "p (permutaciones, dos colas)": round(p, 3)})

display(pd.DataFrame(filas).set_index("variante"))

,d (tamaño de efecto),"p (permutaciones, dos colas)"
variante,,
1. propuesta inicial,0.32,0.542
2. otros atributos,1.17,0.014
3. otros nombres,0.91,0.072


**Lectura.**

* **Observación.** Con la propuesta inicial (nombres, trabajo/hogar) el tamaño de efecto es 0,32 y la prueba de permutaciones da p = 0,54. Si cambiamos solo los atributos por palabras de ciencia y tecnología frente a arte y música, d sube a 1,17 (p = 0,014). Si cambiamos solo los nombres, d vale 0,91 (p = 0,072).
* **Interpretación posible.** El resultado depende de qué listas elegimos: con las mismas herramientas, una variante no muestra asociación y otra sí. Cambiar el eje de los atributos mueve más el resultado que cambiar los nombres.
* **Límite.** Estas tres variantes las armamos nosotros y con ocho palabras por conjunto. Una sola variante no autoriza a decir que el modelo "tiene sesgo", y que no aparezca asociación en una tampoco autoriza a decir que no lo tiene.

## 4. Observación, hipótesis, auditoría, consecuencias

| Nivel | Qué es | Ejemplo con esta tabla |
|---|---|---|
| Observación | Lo que midió el cálculo | "En la variante 1, d vale ..." |
| Hipótesis | Una explicación posible | "El corpus asocia con más frecuencia ... " (no verificada) |
| Auditoría | Un procedimiento definido de antemano que pone a prueba la hipótesis | Muchas listas, sets validados, comparación con un corpus de referencia |
| Consecuencias de uso | Qué efecto tendría usar este modelo en una aplicación concreta | Depende de la aplicación: un autocompletado no es un filtro de currículums |

**Para pensar.** ¿Qué variante cambia más el resultado? ¿Qué pasaría con otro modelo (fastText)? ¿Qué listas usaría una auditoría hecha por otra persona?

> _(Escriban acá.)_